In [1]:
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
from datasets import load_dataset
from scipy.stats import spearmanr, pearsonr

/Users/mathieu/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [10]:
import pandas as pd
from datasets import Dataset, DatasetDict

def load_and_clean(path):
    df = pd.read_csv(path)

    # Harmoniser les noms si présents
    rename_map = {
        "sent1": "sentence1",
        "sent2": "sentence2",
    }
    df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

    # Gérer le cas score / gpt_score -> une seule colonne 'score'
    has_score = "score" in df.columns
    has_gpt   = "gpt_score" in df.columns

    if has_score and has_gpt:
        # Choisir la colonne à garder (priorité au 'score' "humain")
        # Si tu préfères l'autre, inverse simplement ces deux lignes
        df = df.drop(columns=["gpt_score"])
    elif has_gpt and not has_score:
        df = df.rename(columns={"gpt_score": "score"})

    # Supprimer tout doublon de nom de colonne au cas où
    df = df.loc[:, ~df.columns.duplicated(keep="first")]

    # Garder seulement les colonnes utiles si elles existent
    keep = [c for c in ["split", "sentence1", "sentence2", "score"] if c in df.columns]
    df = df[keep]

    # Types
    if "score" in df.columns:
        df["score"] = pd.to_numeric(df["score"], errors="coerce")
    return df

train_df = load_and_clean("donnees/finstsb_to_dev.csv")
test_df  = load_and_clean("donnees/finstsb_to_test.csv")

# Conversion en DatasetDict
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "test":  Dataset.from_pandas(test_df,  preserve_index=False),
})

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 1999
    })
})
{'split': 'dev', 'sentence1': 'Merck undertakes no obligation to publicly update any forward looking statements.', 'sentence2': 'Carlyle assumes no obligation to update any forward-looking statements at this time.', 'score': 4}


In [12]:
from datasets import concatenate_datasets

# Concaténer train et test
ds_all = concatenate_datasets([ds["train"], ds["test"]]).shuffle(seed=42)

# Split 3000 / 1000
train_size = 3000
train_data = ds_all.select(range(train_size))
val_data   = ds_all.select(range(train_size, min(len(ds_all), train_size + 1000)))

In [14]:
# Préparation des exemples d'entraînement
train_examples = [
    InputExample(texts=[s1, s2], label=score / 5.0)
    for s1, s2, score in zip(train_data["sentence1"], train_data["sentence2"], train_data["score"])
]

# DataLoader
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16,
    collate_fn=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").smart_batching_collate
)

# Optimiseur
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
train_loss = losses.CosineSimilarityLoss(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Préparation des données de validation
sentences1 = list(val_data["sentence1"])
sentences2 = list(val_data["sentence2"])
human_scores = val_data["score"]  # scores humains à conserver

# Fonction d’évaluation
def evaluate_model(model_name):
    print(f"\n Évaluation du modèle : {model_name}")
    model = SentenceTransformer(model_name)

    emb1 = model.encode(sentences1, convert_to_tensor=True, show_progress_bar=True)
    emb2 = model.encode(sentences2, convert_to_tensor=True, show_progress_bar=True)

    cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()
    spearman_corr, _ = spearmanr(cosine_scores, human_scores)
    pearson_corr, _ = pearsonr(cosine_scores, human_scores)

    print(f"Spearman correlation : {spearman_corr:.4f}")
    print(f"Pearson correlation  : {pearson_corr:.4f}")
    print("-" * 50)


model.safetensors:  92%|#########2| 83.9M/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
# Chargement du modèle pré-entraîné
model2 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encodage des phrases
emb1 = model2.encode(sentences1, convert_to_tensor=True, show_progress_bar=True)
emb2 = model2.encode(sentences2, convert_to_tensor=True, show_progress_bar=True)

# Similarité cosine
cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()

# Corrélations
spearman_corr, _ = spearmanr(cosine_scores, human_scores)
pearson_corr, _ = pearsonr(cosine_scores, human_scores)

print(f"Spearman correlation : {spearman_corr:.4f}")
print(f"Pearson correlation  : {pearson_corr:.4f}")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Spearman correlation : 0.8637
Pearson correlation  : 0.9256


In [17]:
model.train()
for epoch in range(10):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        features, labels = batch

        features = [{k: v.to(model.device) for k, v in f.items()} for f in features]
        labels = labels.to(model.device)

        optimizer.zero_grad()
        loss = train_loss(features, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"Batch Loss": loss.item()})

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        emb1 = model.encode(sentences1, convert_to_tensor=True, show_progress_bar=False, device=model.device)
        emb2 = model.encode(sentences2, convert_to_tensor=True, show_progress_bar=False, device=model.device)
        
        # Prédictions du modèle (similarité cosine)
        cosine_scores = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()

        # Corrélations
        spearman_corr, _ = spearmanr(cosine_scores, human_scores)
        pearson_corr, _ = pearsonr(cosine_scores, human_scores)

        # Normalisation des scores humains pour calcul de la loss
        val_labels = torch.tensor(human_scores, dtype=torch.float32).to(model.device) / 5.0
        val_preds = torch.tensor(cosine_scores, dtype=torch.float32).to(model.device)

        # Loss de validation (MSE entre prédiction et score humain)
        val_loss = torch.nn.functional.mse_loss(val_preds, val_labels).item()

    print(f"Epoch {epoch+1} - Spearman: {spearman_corr:.4f} | Pearson: {pearson_corr:.4f} | Validation Loss: {val_loss:.4f}")
    print("-" * 60)

    model.train()  # repasser en mode entraînement pour l'epoch suivante

    #if epoch == 4:  # on va prendre le 5eme epoch car on a un overfitting à partir de lui
        #model.save("finetuned-minilm-similarity-epoch5", safe_serialization=False)
        #print("Modèle sauvegardé à 5 epochs")

Epoch 1:   2%|▏         | 4/188 [00:35<27:21,  8.92s/it, Batch Loss=0.0334]


KeyboardInterrupt: 